<a href="https://colab.research.google.com/github/JFVadiaz25/Laboratorio2Parte1Auto2/blob/main/RAG_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers pypdf accelerate
!pip install -q PyPDF2 sentence-transformers faiss-cpu transformers accelerate tqdm


print("Instalación completada")


Instalación completada


In [ ]:
import os
import faiss
import numpy as np
from tqdm.auto import tqdm
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os
DATA_PATH = "/content/drive/MyDrive/auto/neumonía"
#definimos función
def extract_text_from_pdfs(folder_path): #funcion que recorre todos los documentos que sean pdf del dataset, abre cada pdf y extrae el texto
    texts = []
    for file in os.listdir(folder_path): # itera sobre los nombres de cada archivo para filtrar el nombre, no la ruta completa
        if file.endswith(".pdf"): #filtra si termina en .pdf
            reader = PdfReader(os.path.join(folder_path, file)) #abre el pdf
            text = "" #inicia un string vacío
            for page in reader.pages: #filtra por páginas
                text += page.extract_text() or ""  #extrae el texto de la página y lo concatena con text
            texts.append({"source": file, "text": text}) #añade a la lista texts el nombre del archivo y el texto concatenado
    return texts

documents = extract_text_from_pdfs(DATA_PATH) #llama a data_path que es nuestra ruta y lee los documentos cargados
print(f"Documentos cargados: {len(documents)}")


Documentos cargados: 10


In [ ]:
def split_text(text, chunk_size=1000, overlap=200): #funcion que toma el texto completo y lo divide en partes mas pequeñas, chuncks
    step = chunk_size - overlap #calcula el tamaño de salgo (desde donde empieza el siguiente fragmento)
    return [text[i:i + chunk_size] for i in range(0, len(text), step)]

docs_chunks = [] #lista donde guardaremos los chunks
for doc in documents: #iteramos sobre cada documento
    for chunk in split_text(doc["text"]): #aplicamos la función
        docs_chunks.append({"source": doc["source"], "text": chunk}) #guardamos en la lista

print(f"Total de fragmentos creados: {len(docs_chunks)}")



Total de fragmentos creados: 947


In [ ]:
embedding_model = "intfloat/multilingual-e5-large" #aquí nombramos el modelo de embedding que queremos probar
embedder = SentenceTransformer(embedding_model) # cargamos el modelo de embedding

texts = [d["text"] for d in docs_chunks] #crea una lista con cada fragmento porque en docs estan con titulo, aquí solo hay fragmentos
embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True) #convierte cada texto a vector numerico(text,convertir a numpy, barra de progreso)
#esto devuelve una matriz de n numero de fragmentos x m numeros que contiene al volverse vector
# Crear índice FAISS
dimension = embeddings.shape[1] #toma como dimension la m
index = faiss.IndexFlatL2(dimension) #faiss busca vectors parecidos
index.add(embeddings) # añade todos los embeddings al faiss y tenemos en index todos los vectores

print(f"✅ FAISS creado con {index.ntotal} vectores.")


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

✅ FAISS creado con 947 vectores.


In [ ]:
qa_pipeline = pipeline( #aqui defnimos el question aswering
    "text2text-generation", #texto de entrada a otro (parafraseo)
    model="google/flan-t5-large", #modelo de qa
    tokenizer="google/flan-t5-large", #modelo que divide el texto en tokens
    max_length=200, #salida máxima
    temperature=0.2, #aleatoriedad
    top_p=0.95, #muestreo de tokens probables
)
print("Modelo Flan-T5 cargado correctamente.")


Device set to use cuda:0


Modelo Flan-T5 cargado correctamente.


In [ ]:
def buscar_contexto(query, k=3): #busca los fragmentos en los que vamos a basar la información
    query_emb = embedder.encode([query], convert_to_numpy=True) #convierte la pregunta en un vector numerico
    distances, indices = index.search(query_emb, k) #compara el vector pregunta con los fragmentos recuperados, me devuelve la distancia de comparación y los indices(posiciones) de los fragmentos
    retrieved_chunks = [texts[i] for i in indices[0]] # recupera los fragmentos originales con los indices que se indicaron
    return " ".join(retrieved_chunks)  # devuelve los fragmentos en un solo bloque de texto

def responder(query): #genera respuesta basada en la pregunta
    contexto = buscar_contexto(query)
    prompt = f"Contexto: {contexto}\n\nPregunta: {query}\n\nRespuesta:" #toma el contexto, prompt
    output = qa_pipeline(prompt)[0]["generated_text"] #pasamos el promst al pipeline deqa
    return output


In [ ]:
pregunta = input(" Ingrese su pregunta ")
respuesta = responder(pregunta)
print("Pregunta:", pregunta)
print("\n Respuesta:", respuesta)


 Ingrese su pregunta ¿cuales son las causas de neumonía?
Pregunta: ¿cuales son las causas de neumonía?

 Respuesta: La neumona puede ser causada por diversos microorganismos, incluyendo bacterias, virus y hongos. La neumona bacteriana es la más comn y suele responder bien a los antibióticos. La neumona viral es frecuente en temporadas de gripe y suele tratarse con cuidados de soporte, mientras que la neumona por hongos es más rara y suele afectar a personas con sistemas inmunológicos debilitados.


Pregunta del usuario
        ↓
 Embedding (vector numérico)
        ↓
 Búsqueda semántica en FAISS
        ↓
 Recupera fragmentos más relevantes
        ↓
 Construye prompt (Contexto + Pregunta)
        ↓
 Modelo generativo (Flan-T5, etc.)
        ↓
 Respuesta final generada
